# Topology-preserving trajectory compression (CPLS)

Douglas–Peucker keeps the points that carry the *shape* of a track. It knows
nothing about the world the track lives in, so on a coastal route it happily
replaces a bend around a headland with a straight line over the land.
**CPLS** (Context-aware Piecewise Linear Segmentation) runs Douglas–Peucker and
then, inside every stretch that Douglas–Peucker would have collapsed, refuses
to drop a point when dropping it would make the simplified track cross a
barrier, here the coastline, that the original track did not cross.

This notebook walks through it on real data, the 24-hour AIS sample that ships
with the repository and a coastline clipped to the same area. Nothing needs to
be downloaded:

1. **The problem**: a vessel whose simplified track cuts across land
2. **Data**: the sample AIS and the sample coastline, and how well they agree
3. **Compress**: `cpls_compress` against plain Douglas–Peucker over the fleet
4. **One vessel up close**: which points CPLS keeps, and why
5. **Choosing ε**: what the threshold buys
6. **With MovingPandas**: `TrajectoryCollection` in, the same type out
7. **Speed**
8. **The full coastline** (optional, needs a download)
9. **Notes**: the decisions behind the implementation

`catraj` builds on [MovingPandas](https://movingpandas.org):

> Graser, A. (2019). *MovingPandas: Efficient Structures for Movement Data in Python.* GI_Forum 2019, 7(1), 54–68. [doi:10.1553/giscience2019_01_s54](https://doi.org/10.1553/giscience2019_01_s54)

The method is Algorithm 1 of:

> Mehri, S., Alesheikh, A. A., Basiri, A. (2021). *A Contextual Hybrid Model for Vessel Movement Prediction.* IEEE Access 9, 45600–45613. [doi:10.1109/ACCESS.2021.3066463](https://doi.org/10.1109/ACCESS.2021.3066463)

Tutorial 2, [`02_cadp_compression.ipynb`](02_cadp_compression.ipynb), covers the
other compression method, CADP, which protects *context variables* instead of
topology.

In [1]:
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")   # see catraj/__init__.py

import time
from pathlib import Path

import geopandas as gpd
import holoviews as hv
import hvplot.pandas  # noqa: F401  -- registers .hvplot
import numpy as np
import pandas as pd
import shapely
from pyproj import Transformer

from catraj.compression import (Barriers, compress, compression_ratio, cpls_compress,
                                cpls_select, douglas_peucker, dtw, segment_crossings)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAMPLE = ROOT / "data" / "sample"
FULL_COAST = ROOT / "data" / "zenodo_1167595" / "C2_European_Coastline"

# ---- settings ---------------------------------------------------------------
CRS = 32630                         # UTM zone 30N: metres, as distance thresholds need
EPS = 1_000.0                       # the Douglas–Peucker threshold, metres
hv.extension("bokeh")               # interactive plots: zoom, pan, hover, click the legend
ORANGE, GREEN, RED, INK, LAND = "#e16f24", "#1a7f37", "#cf222e", "#0d1117", "#d9cfa8"

In [2]:
# The archive's coded column names -> what they measure (from AIS_parameters.txt)
AIS_NAMES = {"sourcemmsi": "mmsi", "t": "unix_time", "navigation": "navigational_status",
             "rateofturn": "rate_of_turn", "speedoverg": "speed_over_ground",
             "courseover": "course_over_ground", "trueheadin": "true_heading",
             "vesselType": "vessel_type"}
ais = gpd.read_file(SAMPLE / "AIS" / "AIS_DLT_UTM.shp").rename(columns=AIS_NAMES)
ais = ais.sort_values(["mmsi", "unix_time"], kind="stable").reset_index(drop=True)
land = gpd.read_file(SAMPLE / "Coastline" / "Coastline_UTM.shp")
barriers = Barriers(land)                       # indexed once, reused for every call below

ais["x"], ais["y"] = ais.geometry.x, ais.geometry.y
groups = {k: g for k, g in ais.groupby("mmsi", sort=False)}
N = len(ais)
print(f"{N:,} positions, {len(groups)} vessels, "
      f"{len(land)} land polygons ({shapely.get_num_coordinates(land.geometry.values).sum():,} vertices)")

106,894 positions, 136 vessels, 175 land polygons (12,049 vertices)


In [3]:
to_merc = Transformer.from_crs(CRS, 3857, always_xy=True)   # web mercator, for the basemap
land_merc = land.to_crs(3857)


def land_layer():
    """The land polygons as a holoviews layer (exteriors; the few holes are ignored)."""
    polys = [{"x": np.asarray(p.exterior.xy[0]), "y": np.asarray(p.exterior.xy[1])}
             for p in land_merc.geometry]
    return hv.Polygons(polys).opts(color=LAND, line_color="#8a7f5a", line_width=0.6,
                                   alpha=0.85)


def merc(x, y):
    mx, my = to_merc.transform(np.asarray(x, float), np.asarray(y, float))
    return pd.DataFrame({"mx": mx, "my": my})


def new_crossings(result):
    """Per vessel, the number of segments of the result that cross land and replace
    a stretch of the original that did not."""
    kept = set(result.index)
    counts = {}
    for k, g in groups.items():
        keep = g.index.isin(kept)
        counts[k] = int(segment_crossings(g.x.to_numpy(), g.y.to_numpy(), barriers,
                                          keep=keep, new_only=True).sum())
    return pd.Series(counts)

---
## 1 · The problem

Douglas–Peucker (DP) drops a point when it lies within ε of the straight line
between its neighbours. That is a statement about *distance*. It says nothing
about what lies between the neighbours: a vessel that rounds a headland with a
bend a little narrower than ε is replaced by a chord over the headland.

The map shows the vessel in the sample whose DP track at ε = 1 km cuts land the
most. The original positions stay at sea; the red segments of the DP
simplification do not. Nothing in the distance test could have prevented it.

In [4]:
dp_all = compress(ais, EPS, by="mmsi")                   # no thresholds: plain Douglas–Peucker
dp_cross = new_crossings(dp_all)
vid = dp_cross.sort_values(ascending=False, kind="stable").index[0]
g = groups[vid]
keep = g.index.isin(dp_all.index)
print(f"vessel {vid}: {len(g):,} positions, DP keeps {keep.sum():,}, "
      f"{dp_cross[vid]} DP segments cross land that the original track does not")

vessel 228017700: 2,579 positions, DP keeps 31, 4 DP segments cross land that the original track does not


In [5]:
x, y = g.x.to_numpy(), g.y.to_numpy()
kept_idx = np.flatnonzero(keep)
bad = np.flatnonzero(segment_crossings(x, y, barriers, keep=keep, new_only=True))
bad_segments = [np.column_stack(to_merc.transform(x[kept_idx[i:i + 2]], y[kept_idx[i:i + 2]]))
                for i in bad]
corners = np.vstack(bad_segments)
(wx0, wy0), (wx1, wy1) = corners.min(axis=0), corners.max(axis=0)
pad = max(4_000, 0.4 * max(wx1 - wx0, wy1 - wy0))
window = dict(xlim=(wx0 - pad, wx1 + pad), ylim=(wy0 - pad, wy1 + pad))

track = merc(x, y)
layers = [hv.element.tiles.OSM().opts(alpha=0.5), land_layer(),
          track.hvplot.paths("mx", "my", color="#777777", line_width=1, label="original track"),
          track.hvplot.points("mx", "my", color="#777777", size=7, label="original positions"),
          track.iloc[kept_idx].hvplot.paths("mx", "my", color=RED, line_width=1.6,
                                            label="Douglas–Peucker, ε = 1 km"),
          track.iloc[kept_idx].hvplot.points("mx", "my", color=INK, size=30, label="kept points")]
layers += [hv.Path([s]).opts(color=RED, line_width=5, alpha=0.6) for s in bad_segments]
hv.Overlay(layers).opts(width=900, height=560, xaxis=None, yaxis=None, legend_position="top_right",
                        title=f"vessel {vid}: the thick red segments cross land", **window)

:Overlay
   .Tiles.I                                          :Tiles   [x,y]
   .Polygons.I                                       :Polygons   [x,y]
   .Path.Original_track                              :Path   [mx,my]
   .Points.Original_positions                        :Points   [mx,my]
   .Path.Douglas_en_dash_Peucker_comma_ε_equals_1_km :Path   [mx,my]
   .Points.Kept_points                               :Points   [mx,my]
   .Path.I                                           :Path   [x,y]
   .Path.II                                          :Path   [x,y]
   .Path.III                                         :Path   [x,y]
   .Path.IV                                          :Path   [x,y]

---
## 2 · Data

The sample is a 24-hour extract of one public archive:

> Ray, C., Dréo, R., Camossi, E., Jousselme, A.-L., Iphar, C. (2019). *Heterogeneous integrated dataset for Maritime Intelligence, Surveillance, and Reconnaissance.* Data in Brief 25, 104141. [doi:10.1016/j.dib.2019.104141](https://doi.org/10.1016/j.dib.2019.104141)
> Data: Zenodo, [doi:10.5281/zenodo.1167595](https://doi.org/10.5281/zenodo.1167595), record <https://zenodo.org/records/1167595>

| Layer | Folder | What it is | Source | Licence |
|---|---|---|---|---|
| AIS | `data/sample/AIS/` | vessel positions, 2016-02-05 01:00 to 2016-02-06 01:00 UTC | Naval Academy Research Institute (NARI), Brest, France | **[CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)** |
| Coastline | `data/sample/Coastline/` | **land** polygons, clipped to the AIS area plus 5 km | European Environment Agency, "EEA coastline for analysis" (2013), from the same Zenodo record | EEA re-use policy: free to re-use, **acknowledge the source** (copyright holder DG ENV) |

**What the licences require.** *AIS*: attribute NARI and cite the dataset;
**no commercial use**; anything derived from it, including the outputs of this
notebook, is shared under CC BY-NC-SA 4.0. *Coastline*: acknowledge the EEA.
The recipe that cut the coastline from the full archive is in
`data/sample/Coastline/Coastline_parameters.txt`, and the licences are in
[`data/LICENSE.md`](../data/LICENSE.md).

The barriers are the land polygons: a position inside them, or a segment
touching them, is "on land". The coastline is made for analysis at about
1:100,000, so it and the AIS positions do not agree perfectly. The cell
below measures how well they do. Positions on land, and original segments that
already touch land, are real features of the data; CPLS has to live with them
(decision D4 in the notes).

In [6]:
on_land = shapely.intersects(shapely.union_all(land.geometry.values), ais.geometry.values)
orig_hits = sum(int(segment_crossings(g.x.to_numpy(), g.y.to_numpy(), barriers).sum())
                for g in groups.values())
n_segments = sum(len(g) - 1 for g in groups.values())
print(f"positions on land: {on_land.sum()} of {N:,} "
      f"({ais.mmsi[on_land].nunique()} vessels)")
print(f"original consecutive segments touching land: {orig_hits} of {n_segments:,}")

positions on land: 2 of 106,894 (2 vessels)
original consecutive segments touching land: 10 of 106,758


---
## 3 · Compress

`cpls_compress(traj, epsilon, barriers, by=...)` takes the same kind of input
as `compress`, a `GeoDataFrame` of points in trajectory order, a MovingPandas
`Trajectory`, or a `TrajectoryCollection`, and returns the same kind of object
holding only the retained rows, every column intact. Barriers are anything
`Barriers(...)` accepts; wrapping them once, as above, indexes them once for
every vessel. They must be in the same projected CRS as the positions; a
mismatch raises instead of reprojecting silently. CPLS does not use time.

Here: ε = 1 km, one trajectory per vessel (`by="mmsi"`), against plain
Douglas–Peucker, which is `compress` with no thresholds.

How to read the table:

* **points kept** and **compression ratio R**: how much was removed.
* **DTW**: how far the shape moved, over a fixed sample of vessels, the same for
  both methods (dynamic time warping, lower is closer; it is quadratic in
  trajectory length, so it is not computed for every vessel).
* **new crossings**: segments of the result that touch land although the
  stretch of the original they replace does not. This is the quantity CPLS
  guarantees to be zero.
* **vessels affected**: vessels with at least one new crossing.

In [7]:
dp_res = compress(ais, EPS, by="mmsi")
cpls_res = cpls_compress(ais, EPS, barriers, by="mmsi")
results = {"Douglas–Peucker": dp_res, "CPLS": cpls_res}

rng = np.random.default_rng(0)
eligible = [k for k, g in groups.items() if 100 <= len(g) <= 5000]
DTW_SAMPLE = sorted(rng.choice(eligible, size=min(50, len(eligible)), replace=False))


def dtw_km(result):
    kept = set(result.index)
    total = 0.0
    for k in DTW_SAMPLE:
        g = groups[k]
        xy = np.column_stack([g.x, g.y])
        total += dtw(xy, xy[g.index.isin(kept)])
    return total / 1000


rows = []
for label, res in results.items():
    crossings = new_crossings(res)
    rows.append({"method": label, "points kept": len(res),
                 "R (%)": round(compression_ratio(len(res), N), 2),
                 "DTW (km)": round(dtw_km(res)),
                 "new crossings": int(crossings.sum()),
                 "vessels affected": int((crossings > 0).sum())})
print(f"{N:,} points, {len(groups)} vessels, ε = {EPS:,.0f} m; "
      f"DTW over {len(DTW_SAMPLE)} sample vessels")
pd.DataFrame(rows).set_index("method")

106,894 points, 136 vessels, ε = 1,000 m; DTW over 45 sample vessels


,points kept,R (%),DTW (km),new crossings,vessels affected
method,,,,,
Douglas–Peucker,547,99.49,129033,31,19
CPLS,586,99.45,126384,0,0


CPLS keeps more points than Douglas–Peucker (the extra ones are the price of
staying on the right side of the coast) and introduces no new crossing. The
result is always a superset of the Douglas–Peucker result, which the cell below
confirms.

In [8]:
print("CPLS keeps every Douglas–Peucker point:", set(dp_res.index) <= set(cpls_res.index))
print("added by CPLS:", len(cpls_res) - len(dp_res))

CPLS keeps every Douglas–Peucker point: True
added by CPLS: 39


---
## 4 · One vessel up close

The vessel for which CPLS retains the most points for topology, at ε = 1 km.
`cpls_select` works on arrays and returns a `CplsSelection`: `dp` is the
Douglas–Peucker choice, `topology` the points stage 2 added, `mask` their union.
Both panels below show the same stretch of coast and are linked, so zooming one
zooms the other.

In [9]:
per_vessel = {k: cpls_select(g.x.to_numpy(), g.y.to_numpy(), EPS, barriers)
              for k, g in groups.items()}
vid = max(per_vessel, key=lambda k: (per_vessel[k].topology.sum(), len(groups[k])))
g, sel = groups[vid], per_vessel[vid]
print(f"vessel {vid}: {len(g):,} positions\n")
print(sel.summary())

vessel 228017700: 2,579 positions

2579 points -> 35 kept (98.64% compression)
  Douglas-Peucker      31
  + topology           4


In [10]:
x, y = g.x.to_numpy(), g.y.to_numpy()
v = merc(x, y)
topo = np.flatnonzero(sel.topology)

# zoom on the busiest spot: the added point with most other added points within 10 km
near = np.hypot(x[topo][:, None] - x[topo][None, :], y[topo][:, None] - y[topo][None, :]) < 10_000
centre = topo[near.sum(axis=1).argmax()]
cx, cy = to_merc.transform(x[centre], y[centre])
half = 13_000
window = dict(xlim=(cx - half, cx + half), ylim=(cy - half, cy + half))


def panel(title, mask, added=None):
    layers = [hv.element.tiles.OSM().opts(alpha=0.5), land_layer(),
              v.hvplot.paths("mx", "my", color="#777777", line_width=1, label="original track"),
              v.hvplot.points("mx", "my", color="#777777", size=6, label="original positions"),
              v[mask].hvplot.paths("mx", "my", color=RED if added is None else GREEN,
                                   line_width=1.8, label="compressed track"),
              v[sel.dp].hvplot.points("mx", "my", color=INK, size=32, label="kept by DP")]
    if added is not None:
        layers.append(v[added].hvplot.points("mx", "my", color=ORANGE, size=60,
                                              label="added for topology"))
    return hv.Overlay(layers).opts(title=f"{title}: {mask.sum():,} of {len(v):,} points",
                                   width=520, height=520, xaxis=None, yaxis=None,
                                   legend_position="bottom_right", **window)


hv.Layout([panel("Douglas–Peucker", sel.dp), panel("CPLS", sel.mask, sel.topology)]).cols(2)

:Layout
   .Overlay.I  :Overlay
      .Tiles.I                   :Tiles   [x,y]
      .Polygons.I                :Polygons   [x,y]
      .Path.Original_track       :Path   [mx,my]
      .Points.Original_positions :Points   [mx,my]
      .Path.Compressed_track     :Path   [mx,my]
      .Points.Kept_by_DP         :Points   [mx,my]
   .Overlay.II :Overlay
      .Tiles.I                   :Tiles   [x,y]
      .Polygons.I                :Polygons   [x,y]
      .Path.Original_track       :Path   [mx,my]
      .Points.Original_positions :Points   [mx,my]
      .Path.Compressed_track     :Path   [mx,my]
      .Points.Kept_by_DP         :Points   [mx,my]
      .Points.Added_for_topology :Points   [mx,my]

---
## 5 · Choosing ε

ε is Douglas–Peucker's usual distance threshold; CPLS adds no parameter of its
own. A larger ε makes the Douglas–Peucker segments longer, so their chords
cut more corners: plain Douglas–Peucker introduces more crossings and CPLS has
more to repair. The sweep below runs the whole fleet at four values.

* **DP new crossings** (and vessels): new crossings in the plain Douglas–Peucker result.
* **CPLS added**: points CPLS retains on top of Douglas–Peucker.
* **R**: compression ratio of each.
* **CPLS new crossings**: zero at every ε, by construction.

In [11]:
sweep = []
for eps in (100.0, 500.0, 1_000.0, 5_000.0):
    dp = compress(ais, eps, by="mmsi")
    cp = cpls_compress(ais, eps, barriers, by="mmsi")
    dp_c, cp_c = new_crossings(dp), new_crossings(cp)
    sweep.append({"ε (m)": int(eps), "DP kept": len(dp), "CPLS kept": len(cp),
                  "CPLS added": len(cp) - len(dp),
                  "R DP (%)": round(compression_ratio(len(dp), N), 2),
                  "R CPLS (%)": round(compression_ratio(len(cp), N), 2),
                  "DP new crossings": int(dp_c.sum()), "DP vessels": int((dp_c > 0).sum()),
                  "CPLS new crossings": int(cp_c.sum())})
pd.DataFrame(sweep).set_index("ε (m)")

,DP kept,CPLS kept,CPLS added,R DP (%),R CPLS (%),DP new crossings,DP vessels,CPLS new crossings
ε (m),,,,,,,,
100,1339,1341,2,98.75,98.75,1,1,0
500,687,705,18,99.36,99.34,16,11,0
1000,547,586,39,99.49,99.45,31,19,0
5000,323,431,108,99.70,99.60,57,31,0


---
## 6 · With MovingPandas

`cpls_compress` accepts MovingPandas objects directly and returns the same
type, with every column of the retained points. A `TrajectoryCollection` is
compressed trajectory by trajectory, against one shared, once-indexed `Barriers`.

In [12]:
import movingpandas as mpd

frame = ais.assign(time=pd.to_datetime(ais.unix_time, unit="s"))
busiest = frame.mmsi.value_counts().index[:20]
collection = mpd.TrajectoryCollection(frame[frame.mmsi.isin(busiest)],
                                      traj_id_col="mmsi", t="time")
small = cpls_compress(collection, EPS, barriers)

before = sum(len(tr.df) for tr in collection.trajectories)
after = sum(len(tr.df) for tr in small.trajectories)
print(f"{type(small).__name__}: {len(collection.trajectories)} trajectories, "
      f"{before:,} -> {after:,} points (R = {compression_ratio(after, before):.2f}%)")

traj = small.trajectories[0]
print(type(traj).__name__, traj.id, f"{len(traj.df):,} points")
traj.df[["speed_over_ground", "course_over_ground", "vessel_type"]].head()

TrajectoryCollection: 20 trajectories, 87,493 -> 207 points (R = 99.76%)
Trajectory 215566000 6 points


,speed_over_ground,course_over_ground,vessel_type
time,,,
2016-02-05 03:43:12,6.9,21.3,Cargo
2016-02-05 05:58:26,8.1,54.7,Cargo
2016-02-05 07:21:06,7.7,64.8,Cargo
2016-02-05 07:40:06,9.2,72.0,Cargo
2016-02-05 08:00:09,7.7,53.6,Cargo


---
## 7 · Speed

Douglas–Peucker and CPLS over the whole fleet, the mean of 5 runs. Stage 2 is
only run where a Douglas–Peucker segment's bounding box meets a barrier, which
at sea is almost nowhere, so CPLS costs a few times what Douglas–Peucker does
and stays well under a second here. Absolute times depend on the machine.

In [13]:
timing = []
for eps in (100.0, 500.0, 1_000.0, 5_000.0):
    row = {"ε (m)": int(eps)}
    for name, job in (("DP", lambda e=eps: compress(ais, e, by="mmsi")),
                      ("CPLS", lambda e=eps: cpls_compress(ais, e, barriers, by="mmsi"))):
        times = []
        for _ in range(5):
            t0 = time.perf_counter(); job(); times.append(time.perf_counter() - t0)
        row[f"{name} (s)"] = round(float(np.mean(times)), 3)
    timing.append(row)
print(f"{N:,} points, {len(groups)} vessels, mean of 5 runs")
pd.DataFrame(timing).set_index("ε (m)")

106,894 points, 136 vessels, mean of 5 runs


,DP (s),CPLS (s)
ε (m),,
100,0.123,0.379
500,0.102,0.390
1000,0.094,0.382
5000,0.103,0.489


---
## 8 · The full coastline (optional)

The sample coastline covers the sample's area. For other data, use the whole
EEA layer, 71,514 land polygons for Europe. It is not committed; get it with

```bash
python scripts/download_data.py cpls        # 53 MB, into data/zenodo_1167595/C2_European_Coastline/
```

The cell below runs only if that folder exists. It reads just the polygons
around the AIS positions (a `bbox` read, which is fast), reprojects them from
EPSG:3035 to the CRS of the positions, and uses them as barriers.

In [14]:
src = FULL_COAST / "Europe Coastline (Polygone).shp"
if src.exists():
    xmin, ymin, xmax, ymax = ais.total_bounds
    box = shapely.box(xmin - 5000, ymin - 5000, xmax + 5000, ymax + 5000)
    box_3035 = gpd.GeoSeries([shapely.segmentize(box, 1000)], crs=CRS).to_crs(3035).iloc[0]
    full = gpd.read_file(src, bbox=tuple(box_3035.bounds)).to_crs(CRS)
    full_barriers = Barriers(full)
    cp_full = cpls_compress(ais, EPS, full_barriers, by="mmsi")
    crossings = sum(int(segment_crossings(g.x.to_numpy(), g.y.to_numpy(), full_barriers,
                                          keep=g.index.isin(cp_full.index),
                                          new_only=True).sum())
                    for g in groups.values())
    print(f"full coastline around the sample: {len(full)} polygons, "
          f"{shapely.get_num_coordinates(full.geometry.values).sum():,} vertices (unclipped)")
    print(f"CPLS keeps {len(cp_full):,} points with it "
          f"(sample coastline: {len(cpls_res):,}); new crossings: {crossings}")
else:
    print("full coastline not downloaded; run: python scripts/download_data.py cpls")

full coastline around the sample: 206 polygons, 901,147 vertices (unclipped)
CPLS keeps 586 points with it (sample coastline: 586); new crossings: 0


---
## Notes

CPLS is Douglas–Peucker, then a topological pass inside each segment that
Douglas–Peucker collapsed. The paper leaves room in several places;
`help(catraj.compression.cpls)` lists every decision. These are decided
here, not yet confirmed with the authors:

* **D3, what "consistent" means.** A new segment is accepted only if it does not
  intersect a barrier. A segment that merely *touches* a barrier counts as
  intersecting, which is the cautious reading.
* **D4, existing crossings are tolerated.** If the original track already
  touches a barrier between two points (a position on the mapped land, a
  coastline that is a little off), a new segment replacing that stretch may touch
  it too. Otherwise every such stretch would be frozen. The guarantee is "no
  *new* crossing", which is what `segment_crossings(..., new_only=True)` counts.
* **D5, sidedness is not checked.** Only intersection is tested. If the
  original passes north of a small island and the simplified segment passes
  south of it without touching it, CPLS accepts it.
* **D6, ε is a bound on the Douglas–Peucker segment.** Once stage 2 keeps extra
  points, a dropped point is not guaranteed to be within ε of the segment
  between its final neighbours.
* **D7, the greedy is not minimal.** It goes front to back and can keep a point
  because an intermediate chord crossed land, even where the Douglas–Peucker
  segment itself would have been fine.

Also:

* **Barriers** can be lines (a shoreline) or polygons (land), one geometry or
  many, as geometries, a `GeoSeries` or a `GeoDataFrame`.
* **Your own data**: any trajectories in a projected CRS and any barriers in the
  same CRS work the same way.
* **Not implemented yet**: the two-stage variant of the paper (Algorithm 2) and
  the CLSTM model that builds on it.

**Citing.** Please cite:

* **MovingPandas**, which `catraj` builds on: Graser, A. (2019). *MovingPandas: Efficient Structures for Movement Data in Python.* GI_Forum 2019, 7(1), 54–68. [doi:10.1553/giscience2019_01_s54](https://doi.org/10.1553/giscience2019_01_s54)
* **CPLS**, the method: Mehri, S., Alesheikh, A. A., Basiri, A. (2021). *A Contextual Hybrid Model for Vessel Movement Prediction.* IEEE Access 9, 45600–45613. [doi:10.1109/ACCESS.2021.3066463](https://doi.org/10.1109/ACCESS.2021.3066463)
* **The data**: Ray, C. et al. (2019). Data in Brief 25, 104141. [doi:10.1016/j.dib.2019.104141](https://doi.org/10.1016/j.dib.2019.104141); Zenodo [doi:10.5281/zenodo.1167595](https://doi.org/10.5281/zenodo.1167595)
* **The coastline**: European Environment Agency, "EEA coastline for analysis" (2013), via the Zenodo record above; copyright holder DG ENV, [re-use policy](http://www.eea.europa.eu/legal/copyright).

Anything derived from the AIS data, including the outputs of this notebook,
falls under CC BY-NC-SA 4.0.